In [5]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [8]:
def calculate_npc_threat(role, kekayaan_saat_ini, jumlah_tuduhan, total_polisi):
    # 1. SETUP VARIABEL FUZZY (Sama seperti sebelumnya)
    kekayaan = ctrl.Antecedent(np.arange(0, 10001, 1), 'kekayaan')
    tekanan_sosial = ctrl.Antecedent(np.arange(0, 11, 1), 'tekanan_sosial')
    ancaman = ctrl.Consequent(np.arange(0, 101, 1), 'ancaman')

    # Membership Functions
    kekayaan['miskin'] = fuzz.trimf(kekayaan.universe, [0, 0, 4000])
    kekayaan['menengah'] = fuzz.trimf(kekayaan.universe, [2000, 5000, 8000])
    kekayaan['kaya'] = fuzz.trimf(kekayaan.universe, [6000, 10000, 10000])

    tekanan_sosial['aman'] = fuzz.trimf(tekanan_sosial.universe, [0, 0, 3])
    tekanan_sosial['waspada'] = fuzz.trimf(tekanan_sosial.universe, [2, 5, 8])
    tekanan_sosial['bahaya'] = fuzz.trimf(tekanan_sosial.universe, [6, 10, 10])

    ancaman['rendah'] = fuzz.trimf(ancaman.universe, [0, 0, 40])
    ancaman['sedang'] = fuzz.trimf(ancaman.universe, [30, 50, 70])
    ancaman['tinggi'] = fuzz.trimf(ancaman.universe, [60, 100, 100])

    # 2. RULE BASE (Ditambah logika Role)
    # Jika NPC adalah Gangster, mereka lebih sensitif terhadap tekanan sosial
    # 3. RULE BASE (Lengkap - menutupi semua kemungkinan input)
    rules = [
        # --- BLOK TEKANAN BAHAYA (Pasti Tinggi) ---
        ctrl.Rule(tekanan_sosial['bahaya'], ancaman['tinggi']),

        # --- BLOK KEKAYAAN KAYA ---
        ctrl.Rule(kekayaan['kaya'] & tekanan_sosial['aman'], ancaman['sedang']),
        ctrl.Rule(kekayaan['kaya'] & tekanan_sosial['waspada'], ancaman['tinggi']),

        # --- BLOK KEKAYAAN MENENGAH ---
        ctrl.Rule(kekayaan['menengah'] & tekanan_sosial['aman'], ancaman['rendah']),
        ctrl.Rule(kekayaan['menengah'] & tekanan_sosial['waspada'], ancaman['sedang']),

        # --- BLOK KEKAYAAN MISKIN ---
        ctrl.Rule(kekayaan['miskin'] & tekanan_sosial['aman'], ancaman['rendah']),
        ctrl.Rule(kekayaan['miskin'] & tekanan_sosial['waspada'], ancaman['rendah']),
        
        # --- RULE TAMBAHAN (Sangat Penting) ---
        # Jika tekanan sudah waspada, minimal ancaman jadi 'sedang' buat siapa pun
        ctrl.Rule(tekanan_sosial['waspada'], ancaman['sedang']),
    ]
    
    # 3. KONTROL SISTEM
    tipe_ancaman_ctrl = ctrl.ControlSystem(rules)
    simulasi = ctrl.ControlSystemSimulation(tipe_ancaman_ctrl)

    # Input dari Game Engine
    simulasi.input['kekayaan'] = kekayaan_saat_ini
    
    # MODIFIKASI: Kalau Polisi ada 2, tekanan sosial dirasakan lebih berat (+20%)
    multiplier = 1.2 if total_polisi >= 2 else 1.0
    simulasi.input['tekanan_sosial'] = min(jumlah_tuduhan * multiplier, 10)

    simulasi.compute()
    base_threat = simulasi.output['ancaman']

    # 4. PENYESUAIAN AKHIR BERDASARKAN ROLE
    # Gangster secara alami punya base "kewaspadaan" lebih tinggi
    if role in ['gangster', 'ketua_gangster']:
        final_threat = min(base_threat + 10, 100) # Gangster lebih gampang panik/waspada
    else:
        final_threat = base_threat # Warga biasa lebih santai

    return final_threat

In [9]:
npc_role = 'gangster'
duit = 5000
dituduh_berapa_kali = 3
jumlah_polisi_di_room = 2

hasil = calculate_npc_threat(npc_role, duit, dituduh_berapa_kali, jumlah_polisi_di_room)
print(f"Role: {npc_role} | Polisi: {jumlah_polisi_di_room}")
print(f"Tingkat Ancaman/Kepanikan NPC: {hasil:.2f}%")

Role: gangster | Polisi: 2
Tingkat Ancaman/Kepanikan NPC: 60.00%
